# VAE — Paper-to-Code Mock (Colab)

**Paper:** Auto-Encoding Variational Bayes (Kingma & Welling, 2013) — https://arxiv.org/abs/1312.6114

Medium-hard mock (~30 min). Fill in the `VAE` stub (encoder → (mu, logvar) → reparameterize → decoder) and the closed-form KL, run the 2D generation demo, then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement the VAE (YOUR TASK)

- Encoder MLP → two heads: `mu` and `logvar`.
- `reparameterize`: `z = mu + exp(0.5*logvar) * eps`, `eps ~ N(0,I)` (differentiable!).
- Decoder MLP → reconstruction (same dim as input).
- Closed-form KL of a diagonal Gaussian vs. `N(0,I)`.

In [ ]:
class VAE(nn.Module):
    def __init__(self, data_dim=2, hidden=64, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.enc = nn.Sequential(
            nn.Linear(data_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden, latent_dim)
        self.fc_logvar = nn.Linear(hidden, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, data_dim),
        )

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        # TODO: std = exp(0.5*logvar); eps = randn_like(std); return mu + std*eps
        raise NotImplementedError("fill me in")

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def kl_divergence(mu, logvar):
    # TODO: KL(N(mu,sigma^2)||N(0,1)) = -0.5*sum(1 + logvar - mu^2 - exp(logvar)), dim=1
    raise NotImplementedError("fill me in")


def vae_loss(recon, x, mu, logvar):
    recon_term = F.mse_loss(recon, x, reduction="none").sum(dim=1)  # Gaussian likelihood
    kl_term = kl_divergence(mu, logvar)
    return (recon_term + kl_term).mean(), recon_term.mean(), kl_term.mean()

## 2. Demonstrate the benefit (2D generation toy task)
Mixture of three 2D Gaussians. Train the VAE, then sample `z ~ N(0,I)`, decode, and check the generated points land near the true cluster centers.

In [ ]:
CENTERS = torch.tensor([[2.0, 0.0], [-2.0, 0.0], [0.0, 2.0]])

def make_gmm(n, noise=0.15, seed=0):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(0, CENTERS.shape[0], (n,), generator=g)
    return CENTERS[idx] + noise * torch.randn(n, 2, generator=g)

def nearest_center_dist(points):
    return torch.cdist(points, CENTERS).min(dim=1).values

torch.manual_seed(0)
data = make_gmm(2000)
model = VAE(data_dim=2, hidden=64, latent_dim=2)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for epoch in range(800):
    perm = torch.randperm(data.shape[0])
    epoch_loss = 0.0
    for i in range(0, data.shape[0], 256):
        xb = data[perm[i:i+256]]
        recon, mu, logvar = model(xb)
        loss, _, _ = vae_loss(recon, xb, mu, logvar)
        opt.zero_grad(); loss.backward(); opt.step()
        epoch_loss += loss.item() * xb.shape[0]
    losses.append(epoch_loss / data.shape[0])

# GENERATE: sample the prior, decode, measure proximity to real clusters
g = torch.Generator().manual_seed(123)
z = torch.randn(2000, model.latent_dim, generator=g)
with torch.no_grad():
    gen = model.decode(z)
gen_d = nearest_center_dist(gen)
real_d = nearest_center_dist(data)
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"real  mean dist-to-nearest-center: {real_d.mean():.3f}")
print(f"gen   mean dist-to-nearest-center: {gen_d.mean():.3f}")
print(f"fraction of generated within 1.0 of a cluster: {(gen_d < 1.0).float().mean():.3f}")

### (optional) Plot real vs. generated points
Needs matplotlib (preinstalled on Colab). The numeric evidence above is the real check; this is just the pretty version.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data[:, 0].tolist(), data[:, 1].tolist(), s=6, alpha=0.3, label="real")
ax.scatter(gen[:, 0].tolist(), gen[:, 1].tolist(), s=6, alpha=0.3, label="generated (z~N(0,I))")
ax.scatter(CENTERS[:, 0].tolist(), CENTERS[:, 1].tolist(), c="k", marker="x", s=120, label="true centers")
ax.legend(); ax.set_title("VAE: prior samples decode to the data distribution")
plt.show()

## 3. Sanity checks

In [ ]:
# 1) reparameterization is differentiable: gradient flows to mu AND logvar
mu = torch.zeros(4, 2, requires_grad=True)
logvar = torch.zeros(4, 2, requires_grad=True)
VAE().reparameterize(mu, logvar).sum().backward()
assert mu.grad is not None and mu.grad.abs().sum() > 0
assert logvar.grad is not None and logvar.grad.abs().sum() > 0

# 2) closed-form KL: q==prior -> 0; KL >= 0
assert torch.allclose(kl_divergence(torch.zeros(5, 3), torch.zeros(5, 3)), torch.zeros(5))
assert (kl_divergence(torch.randn(100, 3), torch.randn(100, 3)) >= -1e-6).all()

# 3) shapes
m = VAE(data_dim=2, latent_dim=2); x = torch.randn(8, 2)
recon, mu_, lv_ = m(x)
assert m.reparameterize(mu_, lv_).shape == (8, 2)
assert recon.shape == x.shape and mu_.shape == lv_.shape == (8, 2)

# 4) prior samples decode to data-like points
assert gen_d.mean().item() < 0.6
assert (gen_d < 1.0).float().mean().item() > 0.85

# 5) loss/ELBO decreased
assert losses[-1] < 0.5 * losses[0]

# 6) logvar -> -inf makes z deterministic (z == mu)
mf = torch.randn(50, 2)
assert torch.allclose(VAE().reparameterize(mf, torch.full((50, 2), -30.0)), mf, atol=1e-4)

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class VAESolution(nn.Module):
    def __init__(self, data_dim=2, hidden=64, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.enc = nn.Sequential(nn.Linear(data_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU())
        self.fc_mu = nn.Linear(hidden, latent_dim)
        self.fc_logvar = nn.Linear(hidden, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, data_dim))

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar


def kl_divergence_solution(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)